[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_02_pca.ipynb)

# S09 · 02 — Reducir dimensiones con PCA

Sprint 9 · M3 (UD 3.10, PCA con mención a t-SNE)

**Contexto TurisData.** Dirección quiere *ver* a sus huéspedes en una sola imagen, pero tenemos 7 variables por persona y no se puede dibujar en 7 dimensiones. **PCA** (análisis de componentes principales) resume muchas variables en pocas "variables nuevas" (componentes) que conservan la mayor parte de la información. Sirve para **visualizar**, para **quitar ruido** y para **acelerar** otros modelos.

**Este cuaderno tiene 4 ejercicios y un mini-reto.** Los ejercicios 1 y 2 son los mismos que los del cuaderno «en clase» (allí, del 1 al 2).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
variables = [c for c in huespedes.columns if c != "id_huesped"]
X = StandardScaler().fit_transform(huespedes[variables])
grupo = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(X)
print(X.shape)
print("Listo ✔")

### 👀 A2 · EJEMPLO — La varianza explicada de cada componente

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué porcentaje de la información explica el componente 1 y cuál el 2?
2. ¿Qué componentes explican menos del 5 % cada uno?
3. ¿Por qué los porcentajes están ordenados de mayor a menor?

In [ ]:
pca_completo = PCA(random_state=SEMILLA).fit(X)
varianza = pca_completo.explained_variance_ratio_
for i, v in enumerate(varianza, start=1):
    print(f"Componente {i}: {v:.1%}")

### ✏️ A3 · EJERCICIO 1 de 4 — La varianza acumulada y el 90 %

**Tu turno:** Calcula en `varianza_acum` la varianza explicada **acumulada** (usa `np.cumsum` sobre `varianza`) y en `n_comp_90` el **número mínimo de componentes** que explican al menos el 90 % de la varianza.

**Debe salir:** `con 5 componentes conservamos el 90 % de la información` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué valen `n_comp_90` y la varianza acumulada con 4 componentes?
2. ¿Por qué con 4 componentes no basta, si 89,8 % casi es 90 %?
3. ¿Qué ahorro supone quedarse con 5 componentes?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: np.cumsum da la acumulada; para el mínimo de componentes, busca la primera posición que llega a 0,90 y recuerda que se cuenta desde 1
varianza_acum = ...
n_comp_90 = ...

assert abs(varianza_acum[-1] - 1) < 1e-9 and len(varianza_acum) == 7, "varianza_acum debe acabar en 1 y tener 7 valores"
assert n_comp_90 == 5, "Revisa: busca el primer componente donde la acumulada llega a 0.90 y cuéntalo desde 1"
print("con", n_comp_90, "componentes conservamos el 90 % de la información")
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Gráficos de la varianza: por componente y acumulada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿En qué componente cruza la curva la línea roja del 90 %?
2. ¿Qué ve el gráfico de barras que no ve el de la curva?
3. ¿Qué quiere decir que la curva «se aplane» al final?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(range(1, 8), varianza)
ax[0].set_title("Varianza explicada por componente")
ax[0].set_xlabel("Componente"); ax[0].set_ylabel("Proporción")
ax[1].plot(range(1, 8), varianza_acum, marker="o")
ax[1].axhline(0.9, color="tab:red", linestyle="--", label="90 %")
ax[1].set_title("Varianza explicada acumulada")
ax[1].set_xlabel("Número de componentes"); ax[1].legend()
plt.tight_layout(); plt.show()

### 👀 A5 · EJEMPLO — Proyectar los huéspedes a 2 componentes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué forma tiene `Z2` y qué significa cada dimensión?
2. ¿Cuánta información se conserva con 2 componentes?
3. ¿Qué se pierde al quedarse con 2 componentes?

In [ ]:
pca2 = PCA(n_components=2, random_state=SEMILLA)
Z2 = pca2.fit_transform(X)
print("Forma:", Z2.shape, "· información conservada:", f"{pca2.explained_variance_ratio_.sum():.1%}")

### ✏️ A6 · EJERCICIO 2 de 4 — El mapa de huéspedes en 2D

**Tu turno:** Dibuja un diagrama de dispersión de `Z2` (columna 0 en el eje X, columna 1 en el eje Y) coloreando cada punto según la variable `grupo` (el segmento de k-means). Guarda la figura en `fig2d`. Pon título y nombres de ejes con el porcentaje de varianza de cada componente.

**Debe salir:** un gráfico con 4 nubes de puntos de colores (una por segmento) y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Se ven los 4 segmentos separados? ¿Cuáles son los más cercanos entre sí?
2. ¿Qué indica sobre el clustering que los segmentos salgan separados en el mapa?
3. ¿Por qué conviene poner el % de varianza en los ejes?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: ax.scatter(Z2[:, 0], Z2[:, 1], c=grupo, cmap="tab10") dibuja los puntos; después ax.set_title, ax.set_xlabel y ax.set_ylabel
fig2d, ax = plt.subplots(figsize=(7, 5))
...

assert len(fig2d.axes[0].collections) >= 1, "fig2d debe contener un scatter"
assert fig2d.axes[0].get_title() != "", "Añade un título al gráfico"
plt.show()
print("✔ Ejercicio 2 correcto")

### 👀 A7 · EJEMPLO — La tabla de cargas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué variables tienen más peso en el componente 1?
2. ¿Qué variable tiene el mayor peso (en valor absoluto) en el componente 2 y con qué signo?
3. Si el signo de un componente puede salir invertido, ¿qué información de la tabla sigue siendo útil?

In [ ]:
cargas = pd.DataFrame(pca2.components_.T, index=variables, columns=["PC1", "PC2"]).round(2)
cargas

### 👀 A8 · EJEMPLO — Las cargas en un gráfico de barras

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué dos variables tienen la carga de PC1 a la izquierda del cero?
2. ¿Qué variable tiene la barra más larga de PC2 hacia la derecha?
3. ¿Cómo se podría resumir el componente 1 en lenguaje de negocio?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
cargas.plot.barh(ax=ax)
ax.set_title("Cargas de las variables en los dos primeros componentes")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()

### ✏️ A9 · EJERCICIO 3 de 4 — Las variables opuestas a la edad

**Tu turno:** Guarda en `opuestas_a_edad` la lista de variables cuya carga en `PC1` tiene **signo contrario** a la de `edad` (es decir, cuando la edad sube en el componente 1, ellas bajan). Usa el DataFrame `cargas`.

**Debe salir:** `el componente 1 opone «mayor, planificador, estancias largas» a «activo y viajero frecuente»` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué dos variables salen y por qué?
2. Si el componente saliera con todos los signos invertidos, ¿cambiaría el resultado?
3. ¿Qué significa en la práctica que `reservas_anio` sea opuesta a la edad?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: compara np.sign(cargas["PC1"]) con np.sign(cargas.loc["edad", "PC1"])
opuestas_a_edad = ...

assert set(opuestas_a_edad) == {"reservas_anio", "pct_gasto_actividades"}, "Deberían salir 2 variables; revisa el signo de la carga de PC1"
print("el componente 1 opone «mayor, planificador, estancias largas» a «activo y viajero frecuente»")
print("✔ Ejercicio 3 correcto")

### 👀 A10 · EJEMPLO — k-means con distinto número de componentes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Qué ARI se obtiene con 2 componentes y con 5?
2. ¿Qué significa un ARI de 1,00?
3. ¿Por qué no se compara la silueta entre 2, 3, 5 y 7 componentes?

In [ ]:
from sklearn.metrics import adjusted_rand_score
Zcompleto = PCA(random_state=SEMILLA).fit_transform(X)
for n in (2, 3, 5, 7):
    et = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(Zcompleto[:, :n])
    print(f"{n} componentes: coincidencia con el clustering original (ARI) = {adjusted_rand_score(grupo, et):.2f}")

### ✏️ A11 · EJERCICIO 4 de 4 — Agrupar con el 90 % de la varianza

**Tu turno:** Entrena `PCA(n_components=n_comp_90)` sobre `X`, guarda el resultado en `Z90` y agrupa `Z90` con k-means (`k=4`, `n_init=10`, `random_state=SEMILLA`). Guarda las etiquetas en `grupo_pca90` y calcula en `ari_pca90` su coincidencia (ARI) con `grupo`.

**Debe salir:** `ARI = 1.00` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Qué forma tiene `Z90` y por qué?
2. ¿Qué ARI sale y qué dice del clustering con 5 componentes?
3. ¿Qué ventaja tiene entonces reducir con PCA antes de agrupar?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A11, **después de intentarlo**.

In [ ]:
# pista: PCA con n_components fijo y fit_transform; la coincidencia entre dos agrupaciones se mide con adjusted_rand_score
Z90 = ...
grupo_pca90 = ...
ari_pca90 = ...

assert Z90.shape == (1200, n_comp_90), "Z90 debe tener n_comp_90 columnas"
assert ari_pca90 > 0.8, "Con el 90 % de la varianza los grupos deberían ser casi los mismos (ARI > 0,8)"
print(f"ARI = {ari_pca90:.2f}")
print("✔ Ejercicio 4 correcto")

### 👀 A12 · EJEMPLO — t-SNE de los huéspedes

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Cuántas islas ves y coinciden con los segmentos?
2. ¿Se puede decir que dos islas lejanas son «más distintas» que dos cercanas?
3. ¿Para qué sirve t-SNE y para qué no?

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=30, random_state=SEMILLA, init="pca").fit_transform(X)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(tsne[:, 0], tsne[:, 1], c=grupo, cmap="tab10", s=8, alpha=0.7)
ax.set_title("t-SNE de los huéspedes (solo para explorar)")
ax.set_xlabel("dimensión 1 (sin unidades)"); ax.set_ylabel("dimensión 2 (sin unidades)")
plt.show()

### ✏️ A13 · MINI-RETO — ¿Valen 2 componentes?

**Tu turno:** Un compañero te dice: *"con 2 componentes ya vale, quita los demás del modelo"*. Redacta en 3-4 líneas (en una celda de texto) si estás de acuerdo para **(a)** dibujar los huéspedes y **(b)** entrenar un modelo predictivo que necesite precisión. Apóyate en `varianza_acum`.

**Debe salir:** la varianza acumulada con cada número de componentes y ✔ Mini-reto correcto (tu texto de 3-4 líneas queda guardado en `respuesta_reto`)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A13).

1. ¿Estás de acuerdo con «con 2 componentes ya vale» para dibujar?
2. ¿Y para un modelo predictivo que necesite precisión?
3. ¿Qué cifra de `varianza_acum` apoya mejor tu respuesta?

In [ ]:
# pista: mira varianza_acum: con 2 componentes tienes el 69 %; ¿qué te hace falta para dibujar? ¿y para predecir con precisión?
print(np.round(varianza_acum, 3))
respuesta_reto = "..."   # escribe aquí tus 3-4 líneas entre comillas

assert len(respuesta_reto.split()) >= 25, "Escribe tus 3-4 líneas (al menos unas 25 palabras) apoyándote en varianza_acum"
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, el escalado, `PCA`, `KMeans` y la silueta.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `huespedes.csv` (1 200 huéspedes), guarda en `variables` las 7 variables de comportamiento (sin `id_huesped`) y las estandariza en la matriz `X` (PCA se hace **siempre** con variables escaladas).
    - Agrupa los huéspedes en 4 segmentos con k-means (`grupo`, una etiqueta de 0 a 3 por huésped) para poder colorear después los gráficos.
- **Qué debe salir:**
    - `(1200, 7)` (1 200 huéspedes y 7 variables) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno. Los 4 segmentos son los del cuaderno de clustering (S09_01).
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La idea de PCA

#### A2 · Ejemplo — La varianza explicada de cada componente

**Analogía.** Imagina una nube de puntos con forma de puro. Para describirla no necesitas los tres ejes de la habitación: te basta **un eje a lo largo del puro** (donde hay más variación) y otro más corto a lo ancho. PCA encuentra esos nuevos ejes:
- el **componente 1** es la dirección con **más variación** de los datos,
- el **componente 2** es la siguiente, **perpendicular** a la primera, y así sucesivamente.

Cada componente es una **mezcla** de las variables originales, y viene con un porcentaje de **varianza explicada** (cuánta información conserva).

- **Qué hace:**
    - `PCA(random_state=SEMILLA).fit(X)` calcula tantos componentes como variables (7).
    - `explained_variance_ratio_` da la proporción de varianza que explica cada componente (`varianza`); el bucle las escribe en porcentaje.
- **Qué debe salir:**
    - Siete líneas: `Componente 1: 43.2%` · `Componente 2: 25.5%` · `Componente 3: 12.6%` · `Componente 4: 8.5%` · `Componente 5: 4.1%` · `Componente 6: 3.4%` · `Componente 7: 2.8%`.
- **Cómo leerlo:**
    - Los porcentajes bajan de mayor a menor y suman 100 %. El primer componente por sí solo ya explica el 43 % de la información de las 7 variables.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué porcentaje de la información explica el componente 1 y cuál el 2?* → **43,2 %** y **25,5 %**.
2. *¿Qué componentes explican menos del 5 % cada uno?* → Del 5 al 7: **4,1 %, 3,4 % y 2,8 %**.
3. *¿Por qué los porcentajes están ordenados de mayor a menor?* → Porque PCA busca primero la dirección con **más variación** y luego, perpendicular a ella, la siguiente con más variación que quede, y así sucesivamente.

</details>

#### A3 · Ejercicio 1 — La varianza acumulada y el 90 %

- **Qué hace:**
    - `np.cumsum(varianza)` calcula la varianza **acumulada**: el primer valor es el del componente 1, el segundo suma los dos primeros, etc. (`varianza_acum`, 7 valores que acaban en 1).
    - `varianza_acum >= 0.90` da una lista de verdadero/falso; `np.argmax(...)` devuelve la **posición de la primera** casilla verdadera (empezando en 0).
    - `+ 1` convierte esa posición en un número de componentes (se cuenta desde 1), y `int(...)` lo pasa a número normal (`n_comp_90`).
- **Qué debe salir:**
    - `con 5 componentes conservamos el 90 % de la información` y `✔ Ejercicio 1 correcto`. Las acumuladas son 43,2 %, 68,7 %, 81,3 %, 89,8 %, 93,8 %, 97,2 % y 100 %.
- **Si falla:**
    - `varianza_acum debe acabar en 1 y tener 7 valores` → no has usado `np.cumsum(varianza)`.
    - `Revisa: busca el primer componente donde la acumulada llega a 0.90 y cuéntalo desde 1` → te has quedado en 4 (la posición, que empieza en 0) o has contado mal: con 4 componentes se llega al 89,8 %, justo por debajo del 90 %.
    - `NameError: name 'varianza' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `n_comp_90` y la varianza acumulada con 4 componentes?* → `n_comp_90 = 5`; con 4 componentes la acumulada es **89,8 %**.
2. *¿Por qué con 4 componentes no basta, si 89,8 % casi es 90 %?* → Porque el criterio es llegar **al menos** al 90 %: con 4 componentes se queda en 89,8 %, por debajo, y hace falta el quinto (93,8 %).
3. *¿Qué ahorro supone quedarse con 5 componentes?* → Pasar de **7 variables a 5** (un 29 % menos) conservando el **90 %** de la información.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
varianza_acum = np.cumsum(varianza)
n_comp_90 = int(np.argmax(varianza_acum >= 0.90)) + 1
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A4 · Ejemplo — Gráficos de la varianza: por componente y acumulada

- **Qué hace:**
    - A la izquierda, `ax[0].bar(...)` dibuja una barra por componente con su varianza explicada.
    - A la derecha, `ax[1].plot(...)` dibuja la varianza acumulada (`varianza_acum`) y `ax[1].axhline(0.9, ...)` traza la línea roja del 90 %.
- **Qué debe salir:**
    - Dos gráficos. Izquierda: siete barras que bajan desde 0,43 (componente 1) hasta 0,03 (componente 7). Derecha: una curva que sube de 0,43 a 1 y cruza la línea roja del 90 % en el 5.º punto.
- **Cómo leerlo:**
    - La curva de la derecha es la que sirve para decidir: se mira dónde corta la línea del 90 %. El «codo» (donde la curva deja de subir deprisa) está hacia el 3.º–4.º componente.
- **Si falla:**
    - `NameError: name 'varianza_acum' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué componente cruza la curva la línea roja del 90 %?* → En el **5.º** (con 4 queda por debajo, 89,8 %).
2. *¿Qué ve el gráfico de barras que no ve el de la curva?* → Cuánto aporta **cada componente por separado**: las barras bajan rápido (43, 25, 13, 8, 4, 3, 3 %).
3. *¿Qué quiere decir que la curva «se aplane» al final?* → Que los últimos componentes **añaden muy poca información**: pasar de 5 a 7 componentes solo sube del 93,8 % al 100 %.

</details>

## 2. Ver a los huéspedes en 2D

#### A5 · Ejemplo — Proyectar los huéspedes a 2 componentes

- **Qué hace:**
    - `PCA(n_components=2, ...)` se queda solo con los 2 primeros componentes.
    - `fit_transform(X)` calcula los componentes y **proyecta** cada huésped sobre ellos: `Z2` tiene, para cada huésped, sus dos coordenadas (componente 1 y componente 2).
    - El `print` escribe la forma de `Z2` y la información que conservan los dos componentes (la suma de sus varianzas explicadas).
- **Qué debe salir:**
    - `Forma: (1200, 2) · información conservada: 68.7%`.
- **Cómo leerlo:**
    - `(1200, 2)`: 1 200 huéspedes con **2 coordenadas** en lugar de 7 variables. Con ellas sí se puede dibujar. El 68,7 % es la suma de 43,2 % y 25,5 %.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene `Z2` y qué significa cada dimensión?* → **(1200, 2)**: una fila por huésped y dos columnas, sus coordenadas en el componente 1 y el componente 2.
2. *¿Cuánta información se conserva con 2 componentes?* → **68,7 %** (43,2 % + 25,5 %).
3. *¿Qué se pierde al quedarse con 2 componentes?* → El **31,3 %** restante de la variación: lo que ves en el plano es una **sombra** de los datos.

</details>

#### A6 · Ejercicio 2 — El mapa de huéspedes en 2D

- **Qué hace:**
    - `plt.subplots(figsize=(7, 5))` crea la figura (`fig2d`) y sus ejes (`ax`).
    - `ax.scatter(Z2[:, 0], Z2[:, 1], c=grupo, cmap="tab10", ...)` dibuja un punto por huésped (columna 0 de `Z2` en el eje X, columna 1 en el Y), coloreado según su segmento.
    - `ax.set_xlabel(...)` y `ax.set_ylabel(...)` ponen el nombre de cada eje con su porcentaje de varianza (`pca2.explained_variance_ratio_`); `ax.set_title(...)` el título; `ax.legend(...)` la leyenda de segmentos.
- **Qué debe salir:**
    - Un gráfico de dispersión con 4 nubes de puntos bien separadas: el segmento 0 arriba a la izquierda, el 2 arriba a la derecha, el 3 abajo a la izquierda y el 1 abajo a la derecha. Ejes: «Componente 1 (43%)» y «Componente 2 (26%)». Después, `✔ Ejercicio 2 correcto`.
    - **Lectura:** los segmentos se ven **separados** en el plano; PCA permite *comprobar a ojo* que el clustering tiene sentido. Ojo: 2 componentes conservan un ~69 % de la información; lo que ves es una **sombra** de los datos, no toda la verdad.
- **Si falla:**
    - `fig2d debe contener un scatter` → no has dibujado nada en `fig2d` con `ax.scatter(...)` o has llamado a otra figura.
    - `Añade un título al gráfico` → falta `ax.set_title(...)` (el título no puede estar vacío).
    - `NameError: name 'Z2' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Se ven los 4 segmentos separados? ¿Cuáles son los más cercanos entre sí?* → **Sí**, 4 nubes distintas. Los más cercanos son el **0** (arriba a la izquierda) y el **3** (abajo a la izquierda), que casi se tocan.
2. *¿Qué indica sobre el clustering que los segmentos salgan separados en el mapa?* → Que el clustering **tiene sentido**: los huéspedes de un mismo segmento están juntos también en el plano de PCA.
3. *¿Por qué conviene poner el % de varianza en los ejes?* → Porque dice **cuánta información** lleva cada eje: el componente 1 (≈ 43 %) lleva casi el doble que el 2 (≈ 26 %), y entre los dos solo se ve el 69 %.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
fig2d, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(Z2[:, 0], Z2[:, 1], c=grupo, cmap="tab10", s=12, alpha=0.7)
ax.set_xlabel(f"Componente 1 ({pca2.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"Componente 2 ({pca2.explained_variance_ratio_[1]:.0%})")
ax.set_title("Mapa de huéspedes (PCA) coloreado por segmento")
ax.legend(*sc.legend_elements(), title="Segmento")
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. ¿Qué significa cada componente? Las cargas

#### A7 · Ejemplo — La tabla de cargas

**Idea.** Un componente es una receta: `0,45·edad + 0,33·gasto − 0,44·%actividades ...`. Los pesos se llaman **cargas**. Mirar qué variables pesan más (y con qué signo) permite ponerle nombre. *El signo global de un componente es arbitrario (puede salir invertido), lo que importa es qué variables van juntas y cuáles en sentido contrario.*

- **Qué hace:**
    - `pca2.components_` guarda la receta de cada componente (una fila por componente); `.T` la traspone para tener una fila por variable.
    - `pd.DataFrame(..., index=variables, columns=["PC1", "PC2"]).round(2)` la convierte en tabla con una fila por variable (`cargas`).
- **Qué debe salir:**
    - Una tabla de 7 filas y 2 columnas: PC1 → edad 0,45 · gasto_medio_dia 0,33 · noches_medias 0,42 · reservas_anio −0,29 · antelacion_media 0,44 · pct_gasto_actividades −0,44 · viaja_con_ninos 0,20. PC2 → edad 0,37 · gasto 0,55 · noches −0,16 · reservas 0,30 · antelación −0,34 · %actividades −0,14 · niños −0,55.
- **Cómo leerlo:**
    - En PC1 pesan sobre todo `edad`, `antelacion_media`, `noches_medias` (positivas) frente a `pct_gasto_actividades` y `reservas_anio` (negativas). En PC2, el gasto y los niños tienen pesos grandes y de signo contrario.
- **Si falla:**
    - `NameError: name 'pca2' is not defined` → falta ejecutar el ejemplo de proyectar a 2 componentes.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué variables tienen más peso en el componente 1?* → `edad` (0,45), `antelacion_media` (0,44) y `pct_gasto_actividades` (−0,44), seguidas de `noches_medias` (0,42).
2. *¿Qué variable tiene el mayor peso (en valor absoluto) en el componente 2 y con qué signo?* → `gasto_medio_dia`, con **+0,55** (y `viaja_con_ninos` con **−0,55**: se oponen).
3. *Si el signo de un componente puede salir invertido, ¿qué información de la tabla sigue siendo útil?* → Qué variables **van juntas** (mismo signo) y cuáles **en sentido contrario**: eso no cambia aunque se invierta todo el componente.

</details>

#### A8 · Ejemplo — Las cargas en un gráfico de barras

- **Qué hace:**
    - `cargas.plot.barh(ax=ax)` dibuja las cargas de PC1 y PC2 como barras horizontales, una pareja por variable.
    - `ax.axvline(0, ...)` traza la línea del cero: las barras a la izquierda son cargas negativas y a la derecha, positivas.
- **Qué debe salir:**
    - Un gráfico de barras horizontales con 7 variables y 2 barras (PC1 y PC2) en cada una.
- **Cómo leerlo:**
    - Compara los lados de la línea del cero: en PC1, `reservas_anio` y `pct_gasto_actividades` están a un lado y `edad`, `noches_medias` y `antelacion_media` al otro.
- **Si falla:**
    - `NameError: name 'cargas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos variables tienen la carga de PC1 a la izquierda del cero?* → `reservas_anio` (−0,29) y `pct_gasto_actividades` (−0,44).
2. *¿Qué variable tiene la barra más larga de PC2 hacia la derecha?* → `gasto_medio_dia` (+0,55).
3. *¿Cómo se podría resumir el componente 1 en lenguaje de negocio?* → «**Mayor, planificador, de estancias largas** frente a **activo y viajero frecuente**»: sube con edad, antelación y noches y baja con el % de gasto en actividades y el número de reservas.

</details>

#### A9 · Ejercicio 3 — Las variables opuestas a la edad

- **Qué hace:**
    - `np.sign(cargas.loc["edad", "PC1"])` da el **signo** (+1 o −1) de la carga de la edad en PC1 (`signo_edad`).
    - La lista por comprensión recorre las variables (`cargas.index`) y se queda con las que tienen **signo distinto** al de la edad (`opuestas_a_edad`).
- **Qué debe salir:**
    - `el componente 1 opone «mayor, planificador, estancias largas» a «activo y viajero frecuente»` y `✔ Ejercicio 3 correcto`. `opuestas_a_edad` es `['reservas_anio', 'pct_gasto_actividades']`.
- **Si falla:**
    - `Deberían salir 2 variables; revisa el signo de la carga de PC1` → has comparado con otra columna (PC2), has tomado las de signo **igual** o has comparado valores en vez de signos.
    - `NameError: name 'cargas' is not defined` → falta el ejemplo de la tabla de cargas.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué dos variables salen y por qué?* → `reservas_anio` y `pct_gasto_actividades`: tienen carga **negativa** en PC1 (−0,29 y −0,44) mientras la edad la tiene positiva (+0,45).
2. *Si el componente saliera con todos los signos invertidos, ¿cambiaría el resultado?* → **No**: seguirían saliendo esas dos variables, porque se compara el signo **relativo** a la edad, no el absoluto. Por eso es robusto entre versiones de scikit-learn.
3. *¿Qué significa en la práctica que `reservas_anio` sea opuesta a la edad?* → Que, en el componente 1, **los huéspedes mayores reservan menos veces al año** (y los que reservan más son más jóvenes y activos).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
signo_edad = np.sign(cargas.loc["edad", "PC1"])
opuestas_a_edad = [v for v in cargas.index if np.sign(cargas.loc[v, "PC1"]) != signo_edad]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. PCA como paso previo al clustering

#### A10 · Ejemplo — k-means con distinto número de componentes

*Cuidado:* la silueta calculada en espacios de distinta dimensión **no es comparable** directamente; aquí solo comprobamos si los grupos son estables. Para medir cuánto se parecen dos agrupaciones usamos el **ARI** (*adjusted Rand index*): 1 = grupos idénticos; cerca de 0 = no se parecen más que al azar.

- **Qué hace:**
    - `PCA(random_state=SEMILLA).fit_transform(X)` proyecta los huéspedes a los 7 componentes (`Zcompleto`).
    - Para cada `n` en (2, 3, 5, 7): agrupa en 4 con k-means usando solo las `n` primeras columnas (`Zcompleto[:, :n]`) y calcula el ARI con el agrupamiento original (`grupo`).
- **Qué debe salir:**
    - `2 componentes: coincidencia con el clustering original (ARI) = 0.94` · `3 componentes: … = 0.97` · `5 componentes: … = 1.00` · `7 componentes: … = 1.00`.
- **Cómo leerlo:**
    - Con solo 2 componentes (69 % de la información) los grupos ya coinciden en un 94 % con los originales; con 5 son idénticos. Reducir dimensiones **no estropea** el clustering.
- **Si falla:**
    - `NameError: name 'grupo' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué ARI se obtiene con 2 componentes y con 5?* → **0,94** y **1,00**.
2. *¿Qué significa un ARI de 1,00?* → Que los 4 grupos son **idénticos** a los originales: todos los huéspedes caen en el mismo grupo.
3. *¿Por qué no se compara la silueta entre 2, 3, 5 y 7 componentes?* → Porque la silueta depende de las distancias, y las distancias en un espacio de 2 dimensiones **no son comparables** con las de 7: por eso se usa el ARI.

</details>

#### A11 · Ejercicio 4 — Agrupar con el 90 % de la varianza

- **Qué hace:**
    - `PCA(n_components=n_comp_90, ...).fit_transform(X)` se queda con los 5 primeros componentes (`Z90`, de 1200 × 5).
    - `KMeans(n_clusters=4, ...).fit_predict(Z90)` agrupa en 4 con esos 5 componentes (`grupo_pca90`).
    - `adjusted_rand_score(grupo, grupo_pca90)` mide su coincidencia con el agrupamiento original (`ari_pca90`).
- **Qué debe salir:**
    - `ARI = 1.00` y `✔ Ejercicio 4 correcto`.
- **Si falla:**
    - `Z90 debe tener n_comp_90 columnas` → no has usado `n_components=n_comp_90` o has aplicado PCA a otra cosa que `X`.
    - `Con el 90 % de la varianza los grupos deberían ser casi los mismos (ARI > 0,8)` → has agrupado `X` en vez de `Z90`, o has puesto otro `k`.
    - `NameError: name 'n_comp_90' is not defined` → falta el ejercicio 1. `adjusted_rand_score` se importa en el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene `Z90` y por qué?* → **(1200, 5)**: 1 200 huéspedes y 5 componentes, los necesarios para el 90 %.
2. *¿Qué ARI sale y qué dice del clustering con 5 componentes?* → **1,00**: los mismos 4 grupos que con las 7 variables, con dos variables menos.
3. *¿Qué ventaja tiene entonces reducir con PCA antes de agrupar?* → Se trabaja con **menos columnas** (5 en vez de 7) sin cambiar el resultado: en datos con muchas variables, el ahorro de tiempo y de ruido puede ser grande.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
Z90 = PCA(n_components=n_comp_90, random_state=SEMILLA).fit_transform(X)
grupo_pca90 = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(Z90)
ari_pca90 = adjusted_rand_score(grupo, grupo_pca90)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Una mención a t-SNE

#### A12 · Ejemplo — t-SNE de los huéspedes

**t-SNE** (y su primo UMAP) es otra técnica para dibujar datos en 2D. A diferencia de PCA, es **no lineal** y muy buena mostrando grupos, pero: (1) las **distancias entre grupos no significan nada**, (2) el resultado cambia según los parámetros y la semilla y (3) no tiene «cargas» que interpretar. Úsalo para **explorar**, no para **demostrar**.

- **Qué hace:**
    - `TSNE(n_components=2, perplexity=30, random_state=SEMILLA, init="pca").fit_transform(X)` calcula 2 coordenadas por huésped (`perplexity` es cuántos vecinos «mira» cada punto). Tarda unos segundos.
    - `ax.scatter(...)` los dibuja coloreados por segmento (`c=grupo`); los ejes no tienen unidades.
- **Qué debe salir:**
    - Un gráfico con 4 islas de puntos de colores, claramente separadas (más que en el mapa de PCA), y algún pequeño fragmento suelto cerca del centro.
- **Cómo leerlo:**
    - Fíjate en que los ejes no tienen unidades ni porcentajes, y en que la distancia entre islas no se puede interpretar. Con otra semilla o `perplexity`, el dibujo cambiaría.
- **Si falla:**
    - `NameError: name 'X' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas islas ves y coinciden con los segmentos?* → **4** islas de color: coinciden con los 4 segmentos de k-means.
2. *¿Se puede decir que dos islas lejanas son «más distintas» que dos cercanas?* → **No**: en t-SNE las distancias entre grupos **no significan nada**; solo es fiable que los puntos de una misma isla sean parecidos.
3. *¿Para qué sirve t-SNE y para qué no?* → Para **explorar** visualmente; no para **demostrar** nada ni para obtener conclusiones cuantitativas (no tiene cargas y cambia con la semilla).

</details>

## 6. Mini-reto integrador

#### A13 · Mini-reto — ¿Valen 2 componentes?

- **Qué hace:**
    - `print(np.round(varianza_acum, 3))` recuerda la varianza acumulada de 1 a 7 componentes para apoyar la respuesta.
    - `respuesta_reto` es una cadena de texto con tu respuesta de 3-4 líneas: **(a)** ¿valen 2 componentes para dibujar? **(b)** ¿y para entrenar un modelo que necesite precisión?
    - La comprobación solo exige que escribas un texto suficiente (unas 25 palabras): la respuesta de la solución de referencia es **una posible**, no la única.
- **Qué debe salir:**
    - `[0.432 0.687 0.813 0.898 0.938 0.972 1.   ]` y `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - Una buena respuesta cita cifras de `varianza_acum`: con 2 componentes, 68,7 % (**(a)** sí, a modo de mapa); con 5, 93,8 % (**(b)** hacen falta más, porque el 31 % que falta puede contener información útil).
- **Si falla:**
    - `Escribe tus 3-4 líneas (al menos unas 25 palabras) apoyándote en varianza_acum` → `respuesta_reto` sigue siendo `"..."` o es demasiado corta.
    - `NameError: name 'varianza_acum' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Estás de acuerdo con «con 2 componentes ya vale» para dibujar?* → **Sí**: con 2 componentes se conserva el **68,7 %** y alcanza para ver cómo se separan los segmentos.
2. *¿Y para un modelo predictivo que necesite precisión?* → **No necesariamente**: el 31 % que falta puede contener información útil. Hay que comparar la precisión con 2, 5 y todos los componentes.
3. *¿Qué cifra de `varianza_acum` apoya mejor tu respuesta?* → **0,687** (2 componentes) para dibujar y **0,938** (5 componentes) como referencia para el modelo; el 90 % exige 5.

</details>

## Preguntas de reflexión
1. ¿Por qué hay que estandarizar antes de PCA? ¿Qué componente dominaría si no lo hicieras (gasto en euros)?
2. Si los dos primeros componentes explican el 69 %, ¿qué significa que el 31 % restante quede "fuera" del gráfico?
3. ¿Es PCA una técnica supervisada? ¿Puede ayudar a un modelo de cancelaciones? ¿Qué se pierde en interpretabilidad?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. PCA busca las direcciones de **más varianza**, y la varianza depende de las unidades. Sin estandarizar, el gasto en euros (varianza ≈ 3 700) aplastaría a `viaja_con_ninos` (≈ 0,2): el componente 1 sería casi solo el gasto (carga 0,95) y explicaría el 70 % por una razón puramente de escala, no de información.
2. Que ese 31,3 % de la variación **no se ve en el plano**: dos huéspedes que parecen cercanos en el gráfico pueden estar lejos en las otras dimensiones. El gráfico es una sombra útil, pero no demuestra por sí solo que dos huéspedes sean parecidos.
3. **No es supervisada**: no mira ninguna etiqueta (ni «cancela / no cancela»). Puede ayudar a un modelo de cancelaciones con menos variables correlacionadas, menos ruido y más velocidad, pero sin saber qué variables predicen. Se pierde **interpretabilidad**: un componente es una mezcla (`0,45·edad + 0,33·gasto − …`) y ya no se puede decir «sube la edad» o «sube el gasto».

</details>

## Qué has aprendido
- PCA crea ejes nuevos ordenados por la información que conservan.
- La varianza acumulada guía cuántos componentes quedarse.
- Las cargas permiten interpretar los ejes; el signo es arbitrario.
- t-SNE es para explorar visualmente, no para sacar conclusiones cuantitativas.